[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t02_radians.ipynb)

# Why mathematicians measure angles in radians

Aryabhata's sine table used a circle of radius 3438, and 3438 is the circle's 21600 arc-minutes divided by 2π: he
measured the radius along the arc. Roger Cotes measured angles by arc in 1714. The word "radian" first appeared in
print in 1873, in an exam at Queen's College Belfast set by James Thomson.

The idea in one line: **measure a turn by the road the arrow's tip travels.**

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Roll a wheel one radius

Take a wheel of radius one and roll it along a road. When it has rolled a distance of one radius, a point on its rim
has turned through an angle. That angle is **one radian**.

Roll a whole turn, and the wheel travels its whole circumference: 2π radii. So **a full turn is 2π radians**.

In [ ]:
def roll(distance=1.0):
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.axhline(0, color=DIM, lw=2)
    cx = distance
    t = np.linspace(0, 2 * np.pi, 300)
    ax.plot(cx + np.cos(t), 1 + np.sin(t), color=DIM, lw=1.6)
    a = -math.pi / 2 - distance                         # the rim point that started touching the road
    ax.plot([0, distance], [0, 0], color=CORAL, lw=6, solid_capstyle="butt", label=f"road travelled: {distance:.2f} radii")
    arc = np.linspace(-math.pi / 2, a, 200)
    ax.plot(cx + np.cos(arc), 1 + np.sin(arc), color=CORAL, lw=4)
    arrow(ax, -math.pi / 2, color=DIM, lw=1.5, start=(cx, 1))
    arrow(ax, a, color=TEAL, start=(cx, 1))
    ax.set_xlim(-1.4, 8.2); ax.set_ylim(-0.3, 2.3); ax.set_aspect("equal"); ax.grid(False)
    ax.set_title(f"rolled {distance:.2f} radii  →  turned {distance:.2f} radians = {math.degrees(distance):.1f}°")
    ax.legend(loc="upper right"); plt.show()

interact(roll, distance=FloatSlider(value=1.0, min=0, max=2 * math.pi, step=0.05, description="rolled"));

**Notice.** At one radius rolled, the wheel has turned about 57.3°. Drag to the end of the slider: 2π, about 6.28
radii, is exactly one full turn.

## 2 · Degrees and radians

A full turn is 360° and also 2π radians. So **π radians = 180°**, and that one fact converts everything:

* degrees to radians: multiply by π ÷ 180;
* radians to degrees: multiply by 180 ÷ π.

In [ ]:
def as_pi(rad):
    """A radian value as a simple fraction of π when it is one (up to /12)."""
    for q in (1, 2, 3, 4, 6, 12):
        p = rad / math.pi * q
        if abs(p - round(p)) < 1e-9:
            p = int(round(p))
            if p == 0: return "0"
            num = "π" if abs(p) == 1 else f"{abs(p)}π"
            return ("−" if p < 0 else "") + (num if q == 1 else f"{num}/{q}")
    return f"{rad:.4f}"

def convert(degrees=150.0):
    rad = math.radians(degrees)
    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    circle_axes(ax)
    t = np.linspace(0, rad, 200)
    ax.plot(np.cos(t), np.sin(t), color=CORAL, lw=5)
    arrow(ax, rad, color=TEAL)
    ax.set_title(f"{degrees:.0f}° = {as_pi(rad)} rad ≈ {rad:.3f} rad")
    plt.show()

interact(convert, degrees=FloatSlider(value=150, min=-360, max=360, step=15, description="degrees"));

In [ ]:
print(f"{'degrees':>8}  radians")
for d in (0, 30, 45, 60, 90, 120, 135, 150, 180, 270, 360):
    print(f"{d:7d}°  {as_pi(math.radians(d)):>6}")

## 3 · Arc length: l = r θ

On a circle of radius r, an angle of θ radians cuts off an arc of length **l = r θ**. That's the definition again,
just scaled: one radian cuts off one radius of arc.

In [ ]:
def arc_length(r=2.0, theta=1.2):
    fig, ax = plt.subplots(figsize=(5.8, 5.8))
    t = np.linspace(0, 2 * np.pi, 300)
    ax.plot(r * np.cos(t), r * np.sin(t), color=DIM, lw=1.4)
    s = np.linspace(0, theta, 200)
    ax.fill(np.r_[0, r * np.cos(s), 0], np.r_[0, r * np.sin(s), 0], color=CORAL, alpha=0.15)
    ax.plot(r * np.cos(s), r * np.sin(s), color=CORAL, lw=5)
    arrow(ax, 0, color=DIM, lw=1.5, r=r); arrow(ax, theta, color=TEAL, r=r)
    ax.set_xlim(-3.3, 3.3); ax.set_ylim(-3.3, 3.3); ax.set_aspect("equal"); ax.grid(False)
    ax.set_title(f"l = r θ = {r:.1f} × {theta:.2f} = {r * theta:.2f}")
    plt.show()

interact(arc_length, r=FloatSlider(value=2.0, min=0.5, max=3.0, step=0.1, description="r"),
         theta=FloatSlider(value=1.2, min=0.1, max=2 * math.pi, step=0.05, description="θ (rad)"));

**Notice.** Double the radius and the arc doubles. Double the angle and the arc doubles. One multiplication, no 360
anywhere.

## 4 · Radians are just numbers

Because a radian is a length divided by a length, an angle in radians is a plain number. So sin 2 means the sine of
two radians, a turn of about 114.6°. Calculators have a DEG and a RAD button for exactly this reason.

In [ ]:
for v in (1, 2, 3, math.pi):
    print(f"sin {v:.4f} (radians) = {math.sin(v):+.4f}    that's a turn of {math.degrees(v):6.1f}°")

## 5 · Why radians are natural: sin x ≈ x for small x

For a small turn, the vertical projection and the arc are almost the same length. So **sin x is almost x**, when x is in
radians. In degrees it's badly wrong. This is why every formula in calculus, physics and engineering uses radians.

In [ ]:
def small_angles(span=1.0):
    x = np.linspace(-span, span, 400)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.4))
    ax.plot(x, x, color=TEAL, lw=5, alpha=0.6, label="x")
    ax.plot(x, np.sin(x), color=CORAL, label="sin x")
    ax.set_title("x in radians: sin x hugs x near 0"); ax.legend(loc="upper left"); ax.set_xlabel("x (radians)")
    xd = np.degrees(x)
    bx.plot(xd, xd, color=TEAL, lw=5, alpha=0.6, label="x (in degrees)")
    bx.plot(xd, np.sin(np.radians(xd)), color=CORAL, label="sin x")
    bx.set_title("x in degrees: not even close"); bx.legend(loc="upper left"); bx.set_xlabel("x (degrees)")
    plt.tight_layout(); plt.show()
    for v in (0.5, 0.1, 0.01):
        print(f"x = {v:<5} sin x = {math.sin(v):.6f}   gap = {v - math.sin(v):.2e}")

interact(small_angles, span=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="zoom ±"));

**Notice.** Zoom in (make the span small). In radians, the two graphs merge. The gap shrinks like x³ ÷ 6.

## 6 · The identity gallery

**π radians = 180°.** Converting there and back changes nothing.

In [ ]:
deg = lambda r: r * 180 / np.pi
rad = lambda d: d * np.pi / 180
check_identity(lambda x: rad(deg(x)), lambda x: x)
check_identity(lambda x: np.sin(x), lambda x: np.sin(rad(deg(x))));

**l = r θ.** The arc length of a whole turn is the circumference, 2π r.

In [ ]:
r = np.linspace(0.5, 3, 6)
print("r:      ", np.round(r, 2)); print("r × 2π: ", np.round(r * 2 * np.pi, 4)); print("2πr:    ", np.round(2 * np.pi * r, 4))
assert np.allclose(r * 2 * np.pi, 2 * np.pi * r, atol=1e-12)

**sin(x + 2π) = sin x.** A turn of 2π radians is one full turn: the arrow is back where it began.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 600)
fig, ax = plt.subplots(figsize=(9, 3.4))
both_sides(ax, x, np.sin(x + 2 * np.pi), np.sin(x), "sin(x + 2π)  and  sin x"); ax.legend(loc="lower right"); plt.show()
check_identity(lambda x: np.sin(x + 2 * np.pi), lambda x: np.sin(x))
check_identity(lambda x: np.cos(x + 2 * np.pi), lambda x: np.cos(x));

**sin x ≈ x (an approximation, not an identity).** It gets better and better as x shrinks: sin x ÷ x heads to 1.

In [ ]:
for v in (1, 0.1, 0.01, 0.001):
    print(f"x = {v:<6} sin x ÷ x = {math.sin(v) / v:.10f}")

## 7 · On paper: the CBSE way

**Example 1.** Convert 40° 20′ into radians.

1. 20′ is 20/60 of a degree, so 40° 20′ = 40 1/3° = 121/3 degrees.
2. Multiply by π ÷ 180: 121/3 × π/180.
3. **40° 20′ = 121π/540 radians.**

In [ ]:
print(f"121π/540 = {121 * math.pi / 540:.6f}   40°20′ in radians = {math.radians(40 + 20 / 60):.6f}")

**Example 2.** Find the radius of a circle in which a central angle of 60° cuts off an arc of 37.4 cm (use π = 22/7).

1. 60° = π/3 radians.
2. l = r θ, so r = l ÷ θ = 37.4 ÷ (π/3) = 37.4 × 3 ÷ π.
3. With π = 22/7: r = 37.4 × 3 × 7 ÷ 22 = **35.7 cm**.

In [ ]:
print(f"r = {37.4 * 3 * 7 / 22:.2f} cm with π = 22/7;  {37.4 * 3 / math.pi:.2f} cm with the true π")

**Try it.**
1. Convert 25° into radians.
2. A wheel makes 360 revolutions in one minute. Through how many radians does it turn in one second?

<details><summary>Answers</summary>

1. 25 × π/180 = 5π/36.
2. 360 per minute is 6 per second; each is 2π radians, so 12π radians.

</details>

## Check yourself
1. How many radians are in a full turn?
2. About how many degrees is one radian?
3. On a circle of radius 10 cm, how long is the arc cut off by 0.5 radians?

<details><summary>Answers</summary>

1. 2π.
2. About 57.3°, which is 180 ÷ π.
3. l = r θ = 10 × 0.5 = 5 cm.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): Miller, "Earliest uses of some words of mathematics" (r); "Roger
Cotes"; "Aryabhata I". NCERT class 11, chapter 3 (examples 1–2 style). CBSE Mathematics syllabus 2026–27. All
retrieved 2026-10-21.